In [ ]:
import polars as pl


from datetime import datetime

In [6]:
# Open file, add metadata and write to parquet
filename = "results.csv"
with open('data/results.csv', mode='r') as file:
    check = pl.scan_csv(file, has_header = True, separator = ",")
    
df = check.with_columns(
    pl.lit((datetime.now().strftime("%Y-%m-%dT%H:%M:%S"))).alias("date_time_ingested"),
    pl.lit(("kaggle")).alias("source")
)

# Write to parquet 

df.sink_parquet("data/bronze_landing_data.parquet")

In [7]:
#Load the parquet file

df = pl.read_parquet("data/bronze_landing_data.parquet")


In [ ]:
df = df.cast({"home_team": pl.String, 
			"away_team": pl.String, 
			"home_score": pl.Int64, 
			"away_score": pl.Int64,
			"competition": pl.String,
			"stadium": pl.String,
			"city": pl.String, 
            "country": pl.String,
            "neutral": pl.Boolean,
            "world_cup": pl.Boolean,
            "source": pl.String})

df = df.with_columns(
    pl.col("date").str.to_datetime(format="%Y-%m-%d"),
    pl.col("date_time_ingested").str.to_datetime(format="%Y-%m-%dT%H:%M:%S")
    )
df.sink_parquet("data/bronze_raw_data.parquet")

AttributeError: 'DataFrame' object has no attribute 'sink_parquet'

In [10]:
# Load the bronze
df = pl.read_parquet("data/bronze_landing_data.parquet")

# Add the source key, keep the seed key the same
df = df.with_columns(
    pl.struct(["date", "home_team", "away_team"]).hash(seed=0).alias("row_hash")
)

# Clean the string columns
string_cols = ["home_team", "away_team", "competition", "stadium", "city", "country", ]

df = df.with_columns(
    pl.col(c).str.strip_chars().str.to_titlecase() for c in string_cols
)

# # Check for nulls & convert if needed.
# df.filter((pl.col("home_team").str.to_lowercase() == "null") | (pl.col("competition") == "Premier League"))

# check = ["home_team","away_team", "home_score", "away_score"]


In [ ]:
# Null values allowed
required_cols = ["date", "home_team", "away_team", "home_score", "away_score", "competition"] # including competition as it 

violations = df.filter(
    pl.any_horizontal([pl.col(c).is_null() for c in required_cols])
)

violation_expr = pl.any_horizontal([pl.col(c).is_null() for c in required_cols])

violations = df.filter(violation_expr)
clean = df.filter(~violation_expr)

# write to the quarintine files 

violations.write_parquet("data/silver_quarintine_kaggle.parquet")

# Null value logic for allowed nulls

In [ ]:
# Dedup, value standardization & key identification

de_dup = clean.unique(pl.col("row_hash"))

# Value Standerdizations: Country, Stadium, teams, Competition, 

silver_reference_team = pl.DataFrame({"team":["England", "Australia", "Wales", "South Africa", "France", "Ireland", "Scotland", "New Zealand", "Argentina","Italy"]})
silver_reference_country = pl.DataFrame({"team":["England", "Australia", "Wales", "South Africa", "France", "Ireland", "Scotland", "New Zealand", "Argentina","Italy"]})
# Competition needs to be thought through more





competition
str
"""1924–25 New Zealand Tour Of Fr…"
"""1906 Home Nations Championship"""
"""1947–48 Australia Tour Of Grea…"
"""2008 Scotland Tour Of Argentin…"
"""1974 South Africa Rugby Union …"
…
"""2008 End Of Year Tests"""
"""1998 Tri Nations Series"""
"""2010 Autumn Internationals"""


In [26]:
de_dup.group_by("competition").agg([]).write_csv("competition.csv")

In [ ]:
# Write a silver cleaned source & add row metadata 
#append only write

In [ ]:
# Resolution rules

In [ ]:
# Write to the 3nf model
# Add business keys (surrgate keys get added by the table)
#matches, teams, venues, competitions